# Notebook 8 — β_KL Ablation: Is the KL Anchor the Cause?
**CS 590NN | Amogh — Causal-chain Link 1: vary β_KL, watch the trade-off appear/disappear**

## Causal claim being tested

**"The KL anchor *causes* the specificity-success trade-off observed in v3."**

If true: as we increase β_KL from 0 to large values, the trade-off correlation (Spearman ρ between B-success and A-retention) should become MORE NEGATIVE (stronger trade-off). At β_KL = 0, the correlation should disappear (independent edits, no shared budget).

## Why this matters

Notebook 6 showed the trade-off correlationally. Notebook 10 measured its functional form. Notebook 8 manipulates the suspected *cause* and watches the effect change. This is the only experiment in the chain that establishes **causation**, not just association.

## Setup

8 sequential pairs, ROMEtrace only (saves 3× compute), Qwen3-0.6B. Vary β_KL ∈ {0.0, 0.05, 0.1, 0.3, 1.0}. For each β_KL, run the full sequential pipeline and measure Spearman ρ between B-success and A-retention.

## Runtime

~30-40 min on A100 (8 pairs × 5 β_KL values × ~45 sec per pair).

### 8.0 Install (auto-restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping', 'matplotlib', 'scipy'])
print('Restarting...')
os.kill(os.getpid(), 9)

### 8.1 Imports — start here after restart

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb8'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE=='cuda':
    free, tot = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{tot/1e9:.1f} GB free)')

### 8.2 Model + CounterFact

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME, center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True); model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f'Loaded {MODEL_NAME}')

@dataclass
class EditSample:
    idx: int; prompt: str; target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()

def make_sample(i):
    rr = raw[i]['requested_rewrite']
    return EditSample(i, rr['prompt'].format(rr['subject']), ' ' + rr['target_new']['str'], ' ' + rr['target_true']['str'])

# Build 8 disjoint pairs (16 distinct samples). Disjoint to avoid extra confound.
N_PAIRS = 8
pair_indices = [(2*i, 2*i+1) for i in range(N_PAIRS)]
print(f'Pairs ({N_PAIRS}):')
for ai, bi in pair_indices[:3]:
    sa, sb = make_sample(ai), make_sample(bi)
    print(f'  A[{ai}]: {sa.prompt!r} -> {sa.target_new!r}')
    print(f'  B[{bi}]: {sb.prompt!r} -> {sb.target_new!r}')

### 8.3 ROME-trace circuit + edit machinery (same protocol as v3)

In [ ]:
NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The capital of Japan is', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
]

def cache_ref(model, prompts):
    cache = []; model.eval()
    with torch.no_grad():
        for p in prompts:
            tk = model.to_tokens(p); lo = model(tk)
            cache.append((tk, torch.log_softmax(lo[0,-1,:], dim=-1).detach().clone()))
    return cache

def kl_neutral(model, ref_cache):
    total = 0.0
    for tk, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(tk)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def kl_anchor(model, anchor_cache):
    total = 0.0
    for tk, anchor_lp in anchor_cache:
        cur_lp = torch.nn.functional.log_softmax(model(tk)[0,-1,:], dim=-1)
        total = total + (anchor_lp.exp() * (anchor_lp - cur_lp)).sum()
    return total

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id].item()

def measure_full_kl(model, ref_cache):
    model.eval()
    with torch.no_grad(): return kl_neutral(model, ref_cache).item()

def rome_trace_circuit(model, sample, top_k=5):
    """Causal-trace ROME-style: zero each MLP, measure delta in target_new logit. Top-k MLPs."""
    model.eval()
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    with torch.no_grad():
        clean = model(tokens)
        clean_score = clean[0,-1, new_id].item()
    deltas = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        def zero_mlp(v, hook): return torch.zeros_like(v)
        with torch.no_grad():
            patched = model.run_with_hooks(tokens, fwd_hooks=[(hn, zero_mlp)])
        deltas.append((L, abs(patched[0,-1,new_id].item() - clean_score)))
    deltas.sort(key=lambda x: -x[1])
    torch.cuda.empty_cache()
    return [L for L, _ in deltas[:top_k]]

def get_mlp_params(model, mlp_layers):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    return params

def edit(model, sample, params, n_steps, lr, beta_kl, ref_cache, beta_protect=0.0, anchor_cache=None):
    """Edit with optional dual-KL anchor (protect-A term during edit B)."""
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train(); opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if beta_kl > 0: loss = loss + beta_kl * kl_neutral(model, ref_cache)
        if beta_protect > 0 and anchor_cache is not None: loss = loss + beta_protect * kl_anchor(model, anchor_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0); opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Snapshotting weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
ref_cache = cache_ref(model, NEUTRAL)
torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

### 8.4 The β_KL Ablation Loop

For each β_KL ∈ {0.0, 0.05, 0.1, 0.3, 1.0}, run all 8 pairs through the full sequential edit pipeline. Measure trade-off correlation per β_KL.

In [ ]:
BETA_KL_GRID = [0.0, 0.05, 0.1, 0.3, 1.0]
BETA_PROTECT = 0.3   # held constant — same as v3
N_STEPS_A, N_STEPS_B = 5, 3
LR = 5e-3

ROWS_FILE = RESULTS_DIR / f'beta_kl_ablation_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()

for beta_kl in BETA_KL_GRID:
    print(f'\n{"="*78}\nβ_KL = {beta_kl}\n{"="*78}')
    for pair_i, (ai, bi) in enumerate(pair_indices):
        sa, sb = make_sample(ai), make_sample(bi)
        try:
            restore(model, original_state)
            # Discover circuits for A and B
            mlp_A = rome_trace_circuit(model, sa, top_k=5)
            mlp_B = rome_trace_circuit(model, sb, top_k=5)
            # Edit A
            params_A = get_mlp_params(model, mlp_A)
            edit(model, sa, params_A, n_steps=N_STEPS_A, lr=LR, beta_kl=beta_kl, ref_cache=ref_cache)
            p_new_A_postA = measure_p_new(model, sa)
            # Cache A's distribution at A's prompt for the protect term
            with torch.no_grad():
                anchor_cache = []
                tk = model.to_tokens(sa.prompt)
                anchor_cache.append((tk, torch.log_softmax(model(tk)[0,-1,:], dim=-1).detach().clone()))
            # Edit B with protection
            params_B = get_mlp_params(model, mlp_B)
            edit(model, sb, params_B, n_steps=N_STEPS_B, lr=LR, beta_kl=beta_kl, ref_cache=ref_cache,
                 beta_protect=BETA_PROTECT, anchor_cache=anchor_cache)
            p_new_A_postAB = measure_p_new(model, sa)
            p_new_B_postAB = measure_p_new(model, sb)
            retention_A = p_new_A_postAB / max(p_new_A_postA, 1e-6)
            print(f"  pair {pair_i}: succB={p_new_B_postAB:.3f} retA={retention_A:.3f} (postA={p_new_A_postA:.3f} -> postAB={p_new_A_postAB:.3f})")
            rows.append({
                'beta_kl': beta_kl, 'pair_i': pair_i, 'A_idx': ai, 'B_idx': bi,
                'p_new_A_postA': p_new_A_postA,
                'p_new_A_postAB': p_new_A_postAB,
                'p_new_B_postAB': p_new_B_postAB,
                'retention_A': retention_A,
                'retention_A_abs': p_new_A_postAB,
                'success_B': p_new_B_postAB,
                'status': 'ok',
            })
            with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
        except Exception as e:
            print(f'  pair {pair_i} FAILED: {e}')
            rows.append({'beta_kl': beta_kl, 'pair_i': pair_i, 'status': 'failed', 'error': str(e)[:200]})
            torch.cuda.empty_cache()

elapsed = (datetime.now()-started).total_seconds()/60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 8.5 Per-β trade-off correlation

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status')=='ok'])
print(f'OK rows: {len(df)}')

print(f"\n{'beta_kl':>8}  {'spearman_rho':>14}  {'p':>10}  {'mean_retA_abs':>14}  {'mean_succB':>11}  {'n':>4}")
print('-'*80)
rho_results = []
for b in BETA_KL_GRID:
    sub = df[df['beta_kl']==b]
    if len(sub) < 4:
        print(f'{b:>8}  too few rows  n={len(sub)}'); continue
    rho, p = stats.spearmanr(sub['success_B'], sub['retention_A_abs'])
    rho_results.append({'beta_kl': b, 'rho': rho, 'p': p, 'n': len(sub),
                        'mean_retA': sub['retention_A_abs'].mean(),
                        'mean_succB': sub['success_B'].mean()})
    print(f'{b:>8}  {rho:>+14.3f}  {p:>10.4g}  {sub["retention_A_abs"].mean():>14.3f}  {sub["success_B"].mean():>11.3f}  {len(sub):>4}')

### 8.6 Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
betas = [r['beta_kl'] for r in rho_results]
rhos = [r['rho'] for r in rho_results]
ax.plot(betas, rhos, 'o-', lw=2, ms=10, color='#cc3333')
ax.axhline(0, color='gray', lw=0.5)
ax.axhline(-0.41, color='blue', ls='--', lw=1, alpha=0.5, label='v3 baseline ρ = -0.41')
ax.set_xscale('symlog', linthresh=0.05)
ax.set_xlabel('β_KL'); ax.set_ylabel('Spearman ρ (success_B vs retention_A)')
ax.set_title('Does β_KL CAUSE the trade-off? Spearman ρ as a function of β_KL')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig1_beta_ablation.png', dpi=140); plt.show()

# Per-β scatter: succB vs retA at each β
fig, axes = plt.subplots(1, len(BETA_KL_GRID), figsize=(3.2*len(BETA_KL_GRID), 4), sharey=True)
for ax, b in zip(axes, BETA_KL_GRID):
    sub = df[df['beta_kl']==b]
    if len(sub) == 0: continue
    ax.scatter(sub['success_B'], sub['retention_A_abs'], alpha=0.7, s=60, edgecolors='black', lw=0.4)
    rho, p = stats.spearmanr(sub['success_B'], sub['retention_A_abs']) if len(sub)>3 else (np.nan, np.nan)
    ax.set_title(f'β_KL = {b}\nρ={rho:+.2f}'); ax.set_xlabel('success_B')
    ax.grid(alpha=0.3); ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel('retention_A_abs')
fig.suptitle('Per-β scatter: trade-off should strengthen as β_KL increases', y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig2_per_beta_scatter.png', dpi=140); plt.show()

### 8.7 Save & Download

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 8 — β_KL ablation',
    'timestamp': ts, 'model': MODEL_NAME, 'n_pairs_per_beta': N_PAIRS,
    'beta_grid': BETA_KL_GRID, 'beta_protect': BETA_PROTECT,
    'rho_by_beta': rho_results,
    'finding': (
        'If ρ becomes more negative as β_KL increases → KL anchor CAUSES the trade-off. '
        'If ρ stays flat → trade-off is independent of KL strength.'
    ),
}
with open(RESULTS_DIR / f'summary_nb8_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'beta_ablation_df_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))
import shutil
bundle = f'nb8_results_{ts}'; shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')

### What this tells us for the causal chain

**If ρ trends from ~0 at β_KL=0 toward more negative at β_KL=1.0:** Causation established. The KL anchor is the source of the trade-off, and the strength of the trade-off is *tunable* via β_KL. This is a real interventional result.

**If ρ is stable across β_KL values:** The trade-off is NOT caused by the KL anchor. It must come from somewhere else — likely gradient interference or shared parameter overlap. This would force a re-think of the mechanism.

**If ρ is non-monotonic (e.g., strongest at intermediate β_KL):** Most interesting. The trade-off has a regime structure — too little KL = no protection; too much KL = both edits fail; sweet spot in the middle.

Whatever the result, this is the experiment that converts the v3 finding from "correlation under one condition" to "causal mechanism with a tunable knob."